In [1]:
import torch
from nnsight import LanguageModel
from autoencoder import AutoEncoder

NAMES = {
    0 : "gpt2-small_L0_Hcat_z_lr1.20e-03_l11.80e+00_ds24576_bs4096_dc1.00e-06_rsanthropic_rie25000_nr4_v9",
    10 : "gpt2-small_L10_Hcat_z_lr1.20e-03_l11.30e+00_ds24576_bs4096_dc1.00e-05_rsanthropic_rie25000_nr4_v9",
    11: "gpt2-small_L11_Hcat_z_lr1.20e-03_l13.00e+00_ds24576_bs4096_dc3.16e-06_rsanthropic_rie25000_nr4_v9",
    1 : "gpt2-small_L1_Hcat_z_lr1.20e-03_l18.00e-01_ds24576_bs4096_dc1.00e-06_rsanthropic_rie25000_nr4_v5",
    2 : "gpt2-small_L2_Hcat_z_lr1.20e-03_l11.00e+00_ds24576_bs4096_dc1.00e-06_rsanthropic_rie25000_nr4_v4",
    3: "gpt2-small_L3_Hcat_z_lr1.20e-03_l19.00e-01_ds24576_bs4096_dc1.00e-06_rsanthropic_rie25000_nr4_v9",
    4 : "gpt2-small_L4_Hcat_z_lr1.20e-03_l11.10e+00_ds24576_bs4096_dc1.00e-06_rsanthropic_rie25000_nr4_v7",
    5 : "gpt2-small_L5_Hcat_z_lr1.20e-03_l11.00e+00_ds49152_bs4096_dc1.00e-06_rsanthropic_rie25000_nr4_v9",
    6 : "gpt2-small_L6_Hcat_z_lr1.20e-03_l11.10e+00_ds24576_bs4096_dc1.00e-06_rsanthropic_rie25000_nr4_v9",
    7 : "gpt2-small_L7_Hcat_z_lr1.20e-03_l11.10e+00_ds49152_bs4096_dc1.00e-06_rsanthropic_rie25000_nr4_v9",
    8 : "gpt2-small_L8_Hcat_z_lr1.20e-03_l11.30e+00_ds24576_bs4096_dc1.00e-05_rsanthropic_rie25000_nr4_v6",
    9 : "gpt2-small_L9_Hcat_z_lr1.20e-03_l11.20e+00_ds24576_bs4096_dc1.00e-06_rsanthropic_rie25000_nr4_v9"
}

attn_sae_list = []
for layer in range(12):
    auto_encoder_run = NAMES[layer]
    attn_sae = AutoEncoder.load_from_hf(auto_encoder_run, hf_repo="ckkissane/attn-saes-gpt2-small-all-layers")

    attn_sae_list.append(attn_sae)

print("Loaded dictionaries")
        
gpt = LanguageModel("openai-community/gpt2", device_map="cuda:0", dispatch=True)

print("Loaded GPT")

kwargs = {
    "load_in_4bit": True,
    "device_map": "cuda:0",
    "dispatch": True
}

mixtral = LanguageModel("mistralai/Mixtral-8x7B-Instruct-v0.1", **kwargs)

print("Loaded Mixtral in 4bit")

Loaded dictionaries
Loaded GPT


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

Loaded Mixtral in 4bit


In [ ]:
messages = [
    {
        "role": "user", 
        "content": "For the rest of the conversation, return your answer in brackets, e.g. {ANSWER}. Just return an answer, nothing else. What is the capital of France?"
    },
    {
        "role": "assistant", 
        "content": "{Paris}"
    },
    {
        "role": "user", 
        "content": "What is a similar sentence to the following: 'I like pizza and big-ass big-scale ham.'? Return one sentence in brackets, nothing else."
    },
]

tok = mixtral.tokenizer.apply_chat_template(messages, return_tensors="pt")

In [3]:
prompt = "the company spent $ 474 million on buy backs previous year - year , $ 40 million extension with the Marlins in million domestically and $ 128 million worldwide . The romantic comedy exceeded more than $ 30 million . Ċ Ċ Aud itors SUV production ; $ 150 million in its Romeo Engine Plant in Chicago of $ 100 million in used money that is of over $ 50 million. Ċ Ċ Related : 10 million to $ 100 million per year , necess itating to make over $ 395 million worldwide . Did you enjoy have grown to $ 32 million after spikes of ? 21 5 million to $ 39 million , worse than its estimate the Dodgers for $ 430 million . Ċ Ċ Advertisement Token Advertisement Feature activation +0.000 Ċ due to receive $ 3 million in a trust fund triggered of more than $ 1 million , but the House never"

In [4]:
with gpt.trace(prompt):
    activation = gpt.transformer.h[0].attn.c_proj.input[0][0]

    loss, x_reconstruct, acts, l2_loss, l1_loss = attn_sae_list[0](activation)
    acts.save()

You're using a GPT2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [17]:
from circuitsvis.tokens import colored_tokens

tokenized = gpt.tokenizer.encode(prompt)
str_tokens = [gpt.tokenizer.decode(i) for i in tokenized]
colored_tokens(str_tokens, acts[:,:,16][0].tolist())